# Gaussian Mixture Models — Rock Decomposition
## Machine learning for energy applications — Module 3, Exercise 3

In this exercise, we will use **Gaussian Mixture Models (GMMs)** to cluster geochemical rock data. The dataset contains major-oxide weight percentages for various igneous rock samples. Each sample also has a reference rock-type label (`ROCK1`, e.g. KOMATIITE). The label is **never a model input**: the GMM is fitted in a fully **unsupervised** manner on the ten oxides only, and `ROCK1` is used afterwards to compare the components with the named rock types.

GMMs extend K-Means by modelling each cluster as a multivariate Gaussian distribution, which allows:
- **Soft (probabilistic) assignments** — each sample has a probability of belonging to every cluster.
- **Elliptical cluster shapes** — clusters are not restricted to spherical geometry.
- **Model selection via BIC/AIC** — principled way to choose the number of components.

**Topics covered:**
1. Load and explore the dataset
2. Missing values, feature selection and standardization
3. Fit a GMM with an initial guess of 3 components
4. Hard cluster assignments and PCA visualization
5. Soft assignments — probability heatmap
6. Model selection with BIC and AIC
7. Refit with optimal number of components
8. Compare GMM clusters with actual rock types
9. Explore covariance types
10. Summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

---
## 1. Load and Explore the Dataset

Load `rock_multiclass_raw.csv` and inspect its shape, data types, missing values, and the distribution of rock types.

In [ ]:
# Load the dataset
df = pd.read_csv('rock_multiclass_raw.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

In [ ]:
# Check the distribution of rock types
print("Rock type distribution:")
print(df['ROCK1'].value_counts())

plt.figure(figsize=(10, 5))
df['ROCK1'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution of Rock Types')
plt.xlabel('Rock Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for the numeric features
df.describe()

---
## 2. Feature Selection and Standardization

Select only the numeric geochemical features (exclude the `ROCK1` label column) and standardize them using `StandardScaler`.

**Missing values first.** `GaussianMixture` does not accept NaN, and nine of the ten oxide columns have gaps. The next cell drops every row with a missing oxide, plus any row without a `ROCK1` label (needed for the comparison in Section 8), and prints which columns caused the loss and how the rock-type counts change. Imputation is the alternative, but it would invent compositions.

**Exploratory analysis of one fixed dataset.** The scaler is fitted on all remaining rows because we only describe these samples; nothing is predicted for new samples, so there is no training/test split.

In [ ]:
# Rows with a missing oxide cannot be used (GaussianMixture rejects NaN), and the
# comparison in Section 8 needs a reference label, so drop both kinds of row.
feature_cols = [c for c in df.columns if c != 'ROCK1']
n_before = len(df)
n_missing = df[feature_cols + ['ROCK1']].isna().sum()
print("Missing values per column (a row can miss several):")
print(n_missing[n_missing > 0].sort_values(ascending=False).to_string())

counts_before = df['ROCK1'].value_counts()
df = df.dropna(subset=feature_cols + ['ROCK1']).reset_index(drop=True)
print(f"\nRows kept: {len(df)} of {n_before} ({n_before - len(df)} dropped)")
print(pd.DataFrame({'before': counts_before, 'after': df['ROCK1'].value_counts()})
      .sort_values('before', ascending=False))

In [ ]:
# Features: the ten oxides (feature_cols, defined above). ROCK1 is kept aside for Section 8.
X = df[feature_cols].to_numpy()
rock_labels = df['ROCK1'].to_numpy()

print(f"Number of features: {len(feature_cols)}")
print(f"Features: {feature_cols}")
print(f"Shape of X: {X.shape}")

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("After scaling:")
print(f"  Mean (should be ~0): {X_scaled.mean(axis=0).round(2)}")
print(f"  Std  (should be ~1): {X_scaled.std(axis=0).round(2)}")

---
## 3. Fit a GMM with 3 Components (Initial Guess)

As a starting point, fit a `GaussianMixture` with `n_components=3`. We will refine this later using BIC/AIC.

In [ ]:
# Fit GMM with 3 components
gmm3 = GaussianMixture(n_components=3, random_state=42, n_init=5)
gmm3.fit(X_scaled)

print(f"Converged: {gmm3.converged_}")
print(f"Number of iterations: {gmm3.n_iter_}")
print(f"Mean log-likelihood per sample (lower_bound_): {gmm3.lower_bound_:.4f}")
print(f"BIC: {gmm3.bic(X_scaled):.2f}")
print(f"AIC: {gmm3.aic(X_scaled):.2f}")
print(f"\nCluster weights (mixing proportions): {gmm3.weights_.round(4)}")

---
## 4. Hard Assignment — PCA 2D Projection

Use `.predict()` to assign each sample to its most likely cluster. Visualize with a 2D PCA projection.

In [ ]:
# Hard cluster assignments
labels_gmm3 = gmm3.predict(X_scaled)

print("Cluster sizes:")
unique, counts = np.unique(labels_gmm3, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} samples")

In [ ]:
# PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Plot GMM clusters in PCA space
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_gmm3, cmap='viridis',
                      edgecolors='k', alpha=0.7, s=50)
plt.colorbar(scatter, label='GMM Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('GMM (K=3) — Hard Cluster Assignments (PCA Projection)')
plt.tight_layout()
plt.show()

---
## 5. Soft Assignment — Probability Heatmap

Use `.predict_proba()` to obtain the probability that each sample belongs to each cluster. Visualize as a heatmap.

In [ ]:
# Soft assignments (probabilities)
probs = gmm3.predict_proba(X_scaled)

print(f"Shape of probability matrix: {probs.shape}")
print(f"\nFirst 5 samples — cluster probabilities:")
prob_df = pd.DataFrame(probs, columns=[f'Cluster {i}' for i in range(3)])
prob_df.head()

In [ ]:
# Sort samples by hard assignment for a cleaner heatmap
sort_idx = np.argsort(labels_gmm3)
probs_sorted = probs[sort_idx]

plt.figure(figsize=(8, 10))
sns.heatmap(probs_sorted, cmap='YlOrRd', xticklabels=[f'Cluster {i}' for i in range(3)],
            yticklabels=False, cbar_kws={'label': 'Probability'})
plt.xlabel('Cluster')
plt.ylabel('Samples (sorted by hard assignment)')
plt.title('GMM Soft Assignments — Probability Heatmap')
plt.tight_layout()
plt.show()

---
## 6. Model Selection — BIC and AIC

Fit GMMs with `n_components` from 1 to 10 and compute the **BIC** (Bayesian Information Criterion) and **AIC** (Akaike Information Criterion) for each. Lower values indicate a better model.

In [ ]:
# Model selection: BIC and AIC for n_components = 1..10
n_range = range(1, 11)
bics = []
aics = []

for n in n_range:
    gmm_temp = GaussianMixture(n_components=n, random_state=42, n_init=5)
    gmm_temp.fit(X_scaled)
    bics.append(gmm_temp.bic(X_scaled))
    aics.append(gmm_temp.aic(X_scaled))
    print(f"n_components={n:2d}  BIC={bics[-1]:10.2f}  AIC={aics[-1]:10.2f}")

In [ ]:
# Plot BIC and AIC
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(list(n_range), bics, 'o-', label='BIC', color='steelblue', linewidth=2)
ax.plot(list(n_range), aics, 's--', label='AIC', color='darkorange', linewidth=2)

# Mark the optimal points
best_bic_n = list(n_range)[np.argmin(bics)]
best_aic_n = list(n_range)[np.argmin(aics)]
ax.axvline(best_bic_n, color='steelblue', linestyle=':', alpha=0.7, label=f'Best BIC: K={best_bic_n}')
ax.axvline(best_aic_n, color='darkorange', linestyle=':', alpha=0.7, label=f'Best AIC: K={best_aic_n}')

ax.set_xlabel('Number of Components (K)')
ax.set_ylabel('Information Criterion')
ax.set_title('GMM Model Selection — BIC and AIC')
ax.set_xticks(list(n_range))
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nOptimal K by BIC: {best_bic_n}")
print(f"Optimal K by AIC: {best_aic_n}")

---
## 7. Refit with Optimal Number of Components

Choose the optimal number of components based on the BIC curve and refit the model.

In [ ]:
# Refit with the BIC-optimal number of components
optimal_k = best_bic_n
print(f"Refitting GMM with K={optimal_k} components...")

gmm_opt = GaussianMixture(n_components=optimal_k, random_state=42, n_init=10)
gmm_opt.fit(X_scaled)

labels_opt = gmm_opt.predict(X_scaled)

print(f"Converged: {gmm_opt.converged_}")
print(f"BIC: {gmm_opt.bic(X_scaled):.2f}")
print(f"Cluster weights: {gmm_opt.weights_.round(4)}")

print("\nCluster sizes:")
unique, counts = np.unique(labels_opt, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} samples")

In [ ]:
# Visualize the optimal GMM clusters in PCA space
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_opt, cmap='tab10',
                      edgecolors='k', alpha=0.7, s=50)
plt.colorbar(scatter, label='GMM Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title(f'GMM (K={optimal_k}, BIC-optimal) — PCA Projection')
plt.tight_layout()
plt.show()

---
## 8. Compare GMM Clusters with Reference Rock Types

`ROCK1` was not used to fit the model, so we can now compare the components with the reference rock types, using a **cross-tabulation** and the **Adjusted Rand Index (ARI)**. The names are a reference, not a ground truth for oxide structure: some are defined partly by other criteria (gabbro is the coarse-grained intrusive equivalent of basalt; tholeiite is a basalt series), so only partial agreement is expected. Treat each component as a hypothesis about a population in oxide space, not as a rock type.

In [ ]:
# Cross-tabulation: GMM cluster vs actual rock type
ct = pd.crosstab(rock_labels, labels_opt, rownames=['Rock Type'], colnames=['GMM Cluster'])
print("Cross-tabulation of Rock Types vs GMM Clusters:")
ct

In [ ]:
# Heatmap of the cross-tabulation
plt.figure(figsize=(10, 6))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', linewidths=0.5)
plt.title('Rock Type vs GMM Cluster — Cross-Tabulation')
plt.tight_layout()
plt.show()

In [ ]:
# Adjusted Rand Index (ARI) against the reference labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
rock_encoded = le.fit_transform(rock_labels)
print("Label encoding:", {name: code for code, name in enumerate(le.classes_)})

ari = adjusted_rand_score(rock_encoded, labels_opt)
sil = silhouette_score(X_scaled, labels_opt)

print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"  (1.0 = identical partitions, about 0.0 = chance-level agreement)")
print(f"\nSilhouette Score: {sil:.4f}")
print(f"  (higher is better, range -1 to 1)")

In [ ]:
# Side-by-side comparison: reference labels vs GMM clusters in PCA space
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Reference rock types (ROCK1)
for i, rock in enumerate(np.unique(rock_labels)):
    mask = rock_labels == rock
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=rock, alpha=0.7, s=50, edgecolors='k')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('Reference Rock Types (ROCK1)')
axes[0].legend(fontsize=8)

# GMM clusters
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_opt, cmap='tab10',
                          alpha=0.7, s=50, edgecolors='k')
plt.colorbar(scatter, ax=axes[1], label='Cluster')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title(f'GMM Clusters (K={optimal_k})')

plt.suptitle('Reference Rock Types vs GMM Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Explore Covariance Types

The `covariance_type` parameter controls the shape of the Gaussian clusters:
- `'full'` — each cluster has its own general covariance matrix (most flexible)
- `'tied'` — all clusters share the same covariance matrix
- `'diag'` — each cluster has its own diagonal covariance (axis-aligned ellipses)
- `'spherical'` — each cluster has a single variance (spherical clusters, like K-Means)

We compare all four using BIC.

In [ ]:
# Compare covariance types
cov_types = ['full', 'tied', 'diag', 'spherical']
results_cov = []

for cov_type in cov_types:
    gmm_cov = GaussianMixture(n_components=optimal_k, covariance_type=cov_type,
                               random_state=42, n_init=5)
    gmm_cov.fit(X_scaled)
    labels_cov = gmm_cov.predict(X_scaled)
    bic_val = gmm_cov.bic(X_scaled)
    aic_val = gmm_cov.aic(X_scaled)
    sil_val = silhouette_score(X_scaled, labels_cov)
    ari_val = adjusted_rand_score(rock_encoded, labels_cov)
    
    results_cov.append({
        'Covariance Type': cov_type,
        'BIC': bic_val,
        'AIC': aic_val,
        'Silhouette': sil_val,
        'ARI': ari_val
    })
    print(f"{cov_type:12s}  BIC={bic_val:10.2f}  AIC={aic_val:10.2f}  Silhouette={sil_val:.4f}  ARI={ari_val:.4f}")

results_cov_df = pd.DataFrame(results_cov)
results_cov_df

In [ ]:
# Visualize clusters for each covariance type
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, cov_type in zip(axes.ravel(), cov_types):
    gmm_cov = GaussianMixture(n_components=optimal_k, covariance_type=cov_type,
                               random_state=42, n_init=5)
    gmm_cov.fit(X_scaled)
    labels_cov = gmm_cov.predict(X_scaled)
    
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_cov, cmap='viridis',
                         alpha=0.7, s=40, edgecolors='k')
    ax.set_title(f'covariance_type = "{cov_type}"\nBIC = {gmm_cov.bic(X_scaled):.0f}')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle(f'GMM (K={optimal_k}) — Covariance Type Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Summary

In this exercise we:

1. **Loaded and explored** the rock geochemistry dataset containing major-oxide weight percentages for multiple igneous rock types.
2. **Dropped rows with missing values** (printing which columns caused the loss and how many rows remain), **standardized** the ten oxides, and kept the rock-type label out of the model inputs.
3. **Fitted an initial GMM** with K=3 components and visualized hard cluster assignments in PCA space.
4. **Examined soft assignments** using `predict_proba()` and a probability heatmap, showing how GMMs provide richer information than hard clustering.
5. **Performed model selection** using BIC and AIC curves across K=1..10, choosing K by the lowest BIC.
6. **Refitted the model** with the BIC-chosen K and compared the components with the reference rock types via cross-tabulation and Adjusted Rand Index.
7. **Explored covariance types** (`full`, `tied`, `diag`, `spherical`) and compared their BIC, silhouette, and ARI scores.

**Key takeaways:**
- GMMs provide probabilistic (soft) cluster assignments, unlike K-Means which gives only hard assignments.
- BIC and AIC balance fit quality against model complexity. BIC chooses the number of components that best describes the oxide distribution, not the number of rock types.
- The `covariance_type` parameter has a significant impact on cluster shapes and model flexibility.
- Some components line up with compositional end-members while others split or merge named rock types. A component is a hypothesis about structure in the oxide space; naming it a rock type needs further evidence.